# Phase 2: Outfit Compatibility Model Training

아이템 쌍의 호환성을 예측하는 경량 MLP 학습.
- Firestore에서 착용 확정 + 좋아요 데이터 수집
- Positive/Negative pair 생성
- MLP (2×512 → 256 → 1, sigmoid) 학습
- TorchScript export → `compatibility_model.pt`

In [ ]:
!pip install -q transformers torch firebase-admin numpy scikit-learn

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 1. Firestore 데이터 수집

서비스 계정 키가 필요합니다.

In [ ]:
# Google Drive에서 서비스 계정 키 로드
from google.colab import drive
drive.mount('/content/drive')

import firebase_admin
from firebase_admin import credentials, firestore

# 서비스 계정 키 경로 설정
SERVICE_ACCOUNT_KEY = "/content/drive/MyDrive/fashioncloset/service-account-key.json"

cred = credentials.Certificate(SERVICE_ACCOUNT_KEY)
firebase_admin.initialize_app(cred)
db = firestore.client()
print("Firestore connected")

In [ ]:
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import requests
from io import BytesIO
from functools import lru_cache

MODEL_NAME = "patrickjohncyh/fashion-clip"
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
clip_model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
clip_model.eval()

@lru_cache(maxsize=2000)
def get_embedding(image_url: str) -> np.ndarray:
    """URL에서 이미지를 다운로드하고 FashionCLIP 임베딩 추출."""
    res = requests.get(image_url, timeout=10)
    res.raise_for_status()
    img = Image.open(BytesIO(res.content)).convert("RGB")
    inputs = processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        emb = clip_model.get_image_features(**inputs)
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb[0].cpu().numpy().astype(np.float32)

print("FashionCLIP ready for embedding extraction")

In [ ]:
# Firestore에서 착용 기록 수집
positive_pairs = []
all_embeddings = {}  # item_id -> embedding

users_ref = db.collection("users")
users = users_ref.limit(100).stream()  # 최대 100명

for user_doc in users:
    uid = user_doc.id
    
    # 착용 기록에서 같은 날 함께 착용된 아이템 수집
    outfits = db.collection("users").document(uid).collection("outfits").stream()
    
    for outfit_doc in outfits:
        data = outfit_doc.to_dict()
        items = data.get("items", [])
        
        # 각 아이템의 임베딩 수집
        item_embs = []
        for item in items:
            item_id = item.get("id", "")
            image_url = item.get("imageUrl", "")
            if not item_id or not image_url:
                continue
            
            if item_id not in all_embeddings:
                try:
                    emb = get_embedding(image_url)
                    all_embeddings[item_id] = emb
                except Exception:
                    continue
            
            item_embs.append(item_id)
        
        # 같은 outfit 내 아이템 쌍 → positive
        for i in range(len(item_embs)):
            for j in range(i + 1, len(item_embs)):
                positive_pairs.append((item_embs[i], item_embs[j]))

print(f"Positive pairs: {len(positive_pairs)}")
print(f"Unique items: {len(all_embeddings)}")

In [ ]:
# Negative pairs: 랜덤 셔플
all_item_ids = list(all_embeddings.keys())
negative_pairs = []
positive_set = set((a, b) for a, b in positive_pairs) | set((b, a) for a, b in positive_pairs)

target_neg = len(positive_pairs)  # 1:1 비율
attempts = 0
max_attempts = target_neg * 10

while len(negative_pairs) < target_neg and attempts < max_attempts:
    a = random.choice(all_item_ids)
    b = random.choice(all_item_ids)
    if a != b and (a, b) not in positive_set:
        negative_pairs.append((a, b))
        positive_set.add((a, b))
    attempts += 1

print(f"Negative pairs: {len(negative_pairs)}")

## 2. 데이터셋 준비

In [ ]:
# Tensor 데이터셋 생성
X_list = []
y_list = []

for a, b in positive_pairs:
    if a in all_embeddings and b in all_embeddings:
        concat = np.concatenate([all_embeddings[a], all_embeddings[b]])
        X_list.append(concat)
        y_list.append(1.0)

for a, b in negative_pairs:
    if a in all_embeddings and b in all_embeddings:
        concat = np.concatenate([all_embeddings[a], all_embeddings[b]])
        X_list.append(concat)
        y_list.append(0.0)

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.float32)

# Train/Val split (80/20)
indices = np.random.permutation(len(X))
split = int(len(X) * 0.8)
train_idx = indices[:split]
val_idx = indices[split:]

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]

print(f"Train: {len(X_train)}, Val: {len(X_val)}")
print(f"Input dim: {X.shape[1]}")

## 3. 모델 정의 & 학습

In [ ]:
class CompatibilityMLP(nn.Module):
    """경량 MLP: 2×512 → 256 → 1, sigmoid."""
    def __init__(self, input_dim: int = 1024):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


model = CompatibilityMLP(input_dim=X.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.BCELoss()

train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
val_ds = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
EPOCHS = 50
best_auc = 0.0
best_state = None

for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)
        loss = criterion(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(xb)
    train_loss /= len(train_ds)
    
    # Validate
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            pred = model(xb)
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(yb.numpy())
    
    auc = roc_auc_score(all_labels, all_preds)
    
    if auc > best_auc:
        best_auc = auc
        best_state = model.state_dict().copy()
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Loss: {train_loss:.4f} | Val AUC: {auc:.4f} | Best: {best_auc:.4f}")

print(f"\nBest AUC-ROC: {best_auc:.4f}")

## 4. TorchScript Export

In [ ]:
# Best model 로드
model.load_state_dict(best_state)
model.eval()
model.cpu()

# TorchScript 변환
example_input = torch.randn(1, X.shape[1])
traced = torch.jit.trace(model, example_input)

output_path = "/content/compatibility_model.pt"
traced.save(output_path)
print(f"Saved: {output_path}")

# 검증
loaded = torch.jit.load(output_path)
test_out = loaded(example_input)
print(f"Test output: {test_out.item():.4f} (expected 0~1)")

# Google Drive에도 저장
drive_path = "/content/drive/MyDrive/fashioncloset/data/compatibility_model.pt"
traced.save(drive_path)
print(f"Saved to Drive: {drive_path}")

## 5. 서버 배포

`compatibility_model.pt`를 서버의 `data/` 폴더에 복사하면 자동 로드됩니다.
- 모델이 없으면 기존 규칙 기반만 사용 (graceful fallback)
- 환경변수 `COMPATIBILITY_MODEL_PATH`로 경로 오버라이드 가능